In [1]:
import os
import sys

txpipe_dir = os.path.join(os.environ["HOME"], "TXPipe")
sys.path.append(txpipe_dir)

import matplotlib.pyplot as plt
import h5py
import numpy as np
from txpipe.data_types import HDFFile, ShearCatalog, FiducialCosmology
import sacc
import yaml
from pprint import pprint
import time
import ceci
from txpipe.twopoint import TXTwoPoint

%matplotlib inline

/global/cfs/projectdirs/lsst/groups/CL/cl_pipeline_project/conda_envs/txpipe_clp/lib/python3.10/site-packages/ceci/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound


In [ ]:
class TXTwoPointRand(TXTwoPoint):
    name = "TXTwoPointRand"
    inputs = [
        ("random_catalog", HDFFile),
        ("shear_catalog", ShearCatalog),
        ("fiducial_cosmology", FiducialCosmology)
    ]
    
    outputs = [("twopoint_gamma_x", SACCFile)]
    
    config_options = {
          # "binning_scale": "Log"   # idk if i even need this
          "max_sep": 250.0, # Mpc
          "min_sep": 10.0, # Mpc
          "nbins": 15,
          "redshift_bin_edges": [0.4, 0.8, 1.2],
          "richness_bin_edges": [20, 30, 200],
          "units": "arcmin",
          "chunk_rows": 100000,
          "nside": 64
          "pixelization": healpix
          "sparse": True
    }

    def make_random_catalog(self, i):
        # As with the lens catalog version, we add the r_col keyword
        # compare to the parent class
        import treecorr

        if not self.config["use_randoms"]:
            return None

        rancat = treecorr.Catalog(
            self.get_input("binned_random_catalog"),
            ext=f"/randoms/bin_{i}",
            ra_col="ra",
            dec_col="dec",
            ra_units="degree",
            dec_units="degree",
            patch_centers=self.get_input("patch_centers"),
            save_patch_dir=self.get_patch_dir("binned_random_catalog", i),
        )
        return rancat

In [6]:
from txpipe.random_cats import TXRandomCat
from txpipe.auxiliary_maps import TXAuxiliaryLensMaps

In [22]:
photometry_cat = h5py.File('/global/homes/k/kabelo/TXPipe/data/example/inputs/photometry_catalog.hdf5', 'r')
print(photometry_cat['photometry'].keys()) 

<KeysViewHDF5 ['dec', 'extendedness', 'g_mag', 'g_mag_err', 'i_mag', 'i_mag_err', 'id', 'mag_err_g', 'mag_err_i', 'mag_err_r', 'mag_err_u', 'mag_err_y', 'mag_err_z', 'mag_g', 'mag_i', 'mag_r', 'mag_u', 'mag_y', 'mag_z', 'r_mag', 'r_mag_err', 'ra', 'redshift_true', 'shear_1', 'shear_2', 'size_true', 'snr_g', 'snr_i', 'snr_r', 'snr_u', 'snr_y', 'snr_z', 'u_mag', 'u_mag_err', 'y_mag', 'y_mag_err', 'z_mag', 'z_mag_err']>


In [43]:
outdir = "/global/homes/k/kabelo/TXPipe/data/example/outputs"
os.makedirs(outdir, exist_ok=True)

args = {
    "config": {},  # should just pick up the defaults
    "pixelization": "healpix",
    "nside": 2048,
    "photometry_catalog": "/global/homes/k/kabelo/TXPipe/data/example/inputs/photometry_catalog.hdf5",
    "aux_lens_maps": f"{outdir}/aux_maps.hdf5",
}

aux_stage = TXAuxiliaryLensMaps(args)

# Now run the stage
aux_stage.run()

/global/homes/k/kabelo/TXPipe/txpipe/utils/provenance.py:18: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  elif hasattr(module, "__version__"):
/global/homes/k/kabelo/TXPipe/txpipe/utils/provenance.py:19: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  v = module.__version__


In [52]:
aux_maps = h5py.File('global/homes/k/kabelo/TXPipe/data/example/outputs/inprogress_aux_maps.hdf5', 'r')